# H SAF super-obs on CF0360: the new field of view (local companion)

GEOSldas predicts each H SAF super-ob (the mean of the raw H121 obs in one lat/lon cell) as a weighted mean of tile
SFMC, `get_obs_pred` in `clsm_ensupd_upd_routines.F90`. The footprint used for that weighting was changed in
`GEOSldas_GridComp` branch `feature/amfox/hsaf-superobs`:

| | old super-ob (commit `e9c365d`) | new super-ob (commit `9624356`) |
|---|---|---|
| namelist | `FOV = s/√π` (0.141° for s = 0.25°), `FOV_units = 'deg'` | `FOV = 12.5`, `FOV_units = 'km'` (the raw-obs footprint) |
| footprint | uniform disk with the cell's area, the same for every super-ob | Gaussian with covariance **C = FOV²·I + S**, where S is the spatial covariance of the super-ob's own raw obs |

**Why:** a super-ob value is the equal-weight mean of its raw obs, so the consistent prediction is the mean of the raw obs'
own (Gaussian, 12.5 km) footprints. The new operator is the single Gaussian with the same mean and covariance as that
mixture.

This notebook is the **local** companion of `cf0360_hsaf_superob_fov.ipynb` (Discover-only). It uses the same box, grid and
code, and needs no Discover data:

- **obs:** local H121 files (Jan 2020; the Discover case, May 2019, is not on this Mac). Same SW Kansas box, an ascending
  pass at the same time of day (03z window) as the Discover case;
- **tiles:** the CF0360 tile table (`ldas_tilecoord.bin`, from `GEOSldas_diagnostics/test_data`). True tile shapes come
  from a window of the BCS 30″ tile raster around the Kansas box (`tile_raster_kansas.npz`, cut on Discover once);
  outside that window, and if the file is missing, tiles are drawn as their centres of mass;
- **tile field for §6:** SMAP L4 (`SPL4SM_Vv8010`, 9 km tiles) `SFMC_FCST`, mapped to the nearest CF0360 tile. §6 only
  compares operators applied to the *same* field, so any realistic soil-moisture field serves. The checks against
  ObsFcstAna (Discover notebook §6c) are not repeated here.

Each section states its point, shows the evidence, and ends with a **Takeaway**. Runs in about a minute in the `regrid` env.

## 0. Settings and setup

In [ ]:
import datetime as dt

SUPEROB_GRID = 0.25            # super-ob cell size [deg] (Discover runs: 0.25; the retest uses 0.5)
FOV_RAW_KM   = 12.5            # obs_param_nml%FOV of the raw H SAF obs [km]
FOV_OLD_DEG  = SUPEROB_GRID / 3.141592653589793 ** 0.5     # old super-ob FOV (e9c365d rule) [deg]

ANA_TIME = dt.datetime(2020, 1, 5, 3)          # analysis time; window = (ANA_TIME - 1.5 h, ANA_TIME + 1.5 h]
REGION   = (36.5, 38.5, -100.5, -98.0)         # lat0, lat1, lon0, lon1: south-west Kansas (as on Discover)
ZOOM     = (37.25, 37.95, -99.55, -98.70)
MARGIN   = 0.6                                 # [deg] read/compute this far outside REGION
SHOW_SAT = 'metop_b'                           # satellite for single-satellite maps

ERRSTD_PCT = 9.0               # obs_param_nml(56:58)%errstd [% saturation]
POROSITY   = 0.45              # only to express errstd roughly in m3/m3 (errstd/100 * porosity) in section 6

In [ ]:
import re
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import netCDF4 as nc
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from scipy import sparse
from scipy.spatial import cKDTree

%matplotlib inline
T_START = time.time()

def find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / '.git').exists():
            return p
    return start

REPO_ROOT = find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT / 'common' / 'python' / 'io'))
from read_GEOSldas import read_tilecoord

H121 = Path('/Users/amfox/Desktop/ASCAT_SSM_CDR/H121')                          # metop_{a,b,c}/YYYY/MM/*.nc
TILECOORD = Path('/Users/amfox/Desktop/GEOSldas_diagnostics/test_data/land_sweeper/LS_OLv8_M36/output/'
                 'SMAP_EASEv2_M36_GLOBAL/rc_out/M21C_test_CF0360.ldas_tilecoord.bin')
# window of the BCS 30" tile raster (CF0360x6C_CF0360x6C-Pfafstetter.rst), 35.9-39.1N, 101.1-97.4W; optional
TILE_RASTER = Path('/Users/amfox/Desktop/M21C_land_local/superob_fov/tile_raster_kansas.npz')
L4_SFMC = Path('/Users/amfox/Desktop/CYGNSS_operator/example_outputs_M09/SPL4SM_Vv8010.sfmc_state_20191101_0000.nc4')

FIG_DIR = REPO_ROOT / 'projects' / 'M21C_land' / 'figures' / 'superob_fov_v2'
FIG_DIR.mkdir(parents=True, exist_ok=True)

SATS = ['metop_a', 'metop_b', 'metop_c']
SAT_LABEL = {'metop_a': 'Metop-A', 'metop_b': 'Metop-B', 'metop_c': 'Metop-C'}
DT_ASSIM = dt.timedelta(hours=3)

MAPL_RADIUS_KM = 6371.0
KM_PER_DEG = np.pi / 180.0 * MAPL_RADIUS_KM          # km_per_deg in the reader and get_obs_pred
KM2DEG = 1.0 / KM_PER_DEG                            # dist_km2deg, latitude
FAC_SEARCH = 2.0                                     # fac_search_FOV_km
N_LAND = 1188020

# colours used throughout: X = exact mixture, N = new, O = old, R = raw mode
OPC = {'X': 'k', 'N': 'tab:green', 'O': 'tab:purple', 'R': 'tab:blue', 'RM': 'tab:cyan'}

### Tiles, obs reader and tile assignment (copied from the Discover notebook)

Single-precision details are kept, because they reproduce GEOSldas exactly on Discover: tile coordinates in float32,
lat/lon formed as `real(lat_raw)*real(1e-6)`, the window shifted by the reader's 64.184 s TT−UTC offset, the pert-grid
cell computed in float32, and the "first minimum" tie rule.

In [ ]:
tc = read_tilecoord(str(TILECOORD))
assert tc['N_tile'] == N_LAND and (tc['typ'] == 100).all()
COM_LON = tc['com_lon'].astype(np.float32)
COM_LAT = tc['com_lat'].astype(np.float32)
BBOX = {k: tc[k].astype(np.float32) for k in ['min_lon', 'max_lon', 'min_lat', 'max_lat']}
AREA = tc['area'].astype(np.float64)                   # only relative areas matter
TREE = cKDTree(np.c_[COM_LON, COM_LAT].astype(np.float64))

# ---- reader: file selection + QC (read_obs_sm_ASCAT_HSAF)
TT_MINUS_UTC = 64.184
UNIX0 = dt.datetime(1970, 1, 1)
W_LOW = (ANA_TIME - DT_ASSIM / 2 - UNIX0).total_seconds() + TT_MINUS_UTC
W_UP = W_LOW + DT_ASSIM.total_seconds()

def window_files(sat, ana_time=ANA_TIME):
    files = []
    for f in sorted((H121 / sat).glob('*/*/*.nc')):
        start = dt.datetime.strptime(f.name.split('_')[-6], '%Y%m%d%H%M%S')
        if ana_time - DT_ASSIM / 2 - dt.timedelta(hours=1) < start <= ana_time + DT_ASSIM / 2:
            files.append(f)
    return files

def read_h121(f, sat, w_low=W_LOW, w_up=W_UP):
    with nc.Dataset(f) as d:
        d.set_auto_maskandscale(False)
        v = {k: d[k][:] for k in ['latitude', 'longitude', 'time', 'surface_soil_moisture', 'surface_flag',
                                  'processing_flag', 'backscatter40_flag', 'wetland_fraction',
                                  'topographic_complexity', 'subsurface_scattering_probability',
                                  'surface_soil_moisture_sensitivity']}
    s8 = lambda a: a.astype(np.int8).astype(np.int32)
    ssm = v['surface_soil_moisture'].astype(np.int32)
    t = v['time'] * 86400.0
    ok = ((ssm >= 0) & (ssm <= 10000) & (t > w_low) & (t <= w_up)
          & ((v['surface_flag'] & 1) == 0) & ((v['processing_flag'] & 3) == 0)
          & ((v['backscatter40_flag'] & 4) == 0)
          & (s8(v['wetland_fraction']) >= 0) & (s8(v['wetland_fraction']) < 10)
          & (s8(v['topographic_complexity']) >= 0) & (s8(v['topographic_complexity']) < 10)
          & (s8(v['subsurface_scattering_probability']) < 5)
          & (v['surface_soil_moisture_sensitivity'] * 1e-7 > 1.0))
    return pd.DataFrame({'sat': sat,
                         'lat': v['latitude'][ok].astype(np.float32) * np.float32(1e-6),
                         'lon': v['longitude'][ok].astype(np.float32) * np.float32(1e-6),
                         'sm': (ssm[ok] * 0.01 / 100).astype(np.float32)})

# ---- obs -> tile (get_tile_num_for_obs): pert grid 1440 x 1080, nearest centre of mass (|dlon| + |dlat|)
NLON_P, NLAT_P = 1440, 1080
DLON_P, DLAT_P = np.float32(360 / NLON_P), np.float32(180 / NLAT_P)

def pert_cell(lat, lon):
    i = np.ceil((np.asarray(lon, np.float32) - np.float32(-180)) / DLON_P).astype(np.int64)
    j = np.ceil((np.asarray(lat, np.float32) - np.float32(-90)) / DLAT_P).astype(np.int64)
    return np.clip(j, 1, NLAT_P) * NLON_P + np.clip(i, 1, NLON_P)

_key = pert_cell(COM_LAT, COM_LON)
_order = np.argsort(_key, kind='stable')
_sorted_key = _key[_order]

def nearest_tile(lat, lon):
    """candidate tile (0-based, -1 = none in pert cell), |dlon|, |dlat| to it, and whether the obs is outside its bbox"""
    lat, lon = np.asarray(lat, np.float32), np.asarray(lon, np.float32)
    k = pert_cell(lat, lon)
    a, b = np.searchsorted(_sorted_key, k), np.searchsorted(_sorted_key, k, side='right')
    t = np.full(len(lat), -1); dx = np.full(len(lat), np.inf, np.float32); dy = dx.copy()
    for n in np.flatnonzero(b > a):
        cand = _order[a[n]:b[n]]
        ddx, ddy = np.abs(lon[n] - COM_LON[cand]), np.abs(lat[n] - COM_LAT[cand])
        best = np.argmin(ddx + ddy)
        t[n], dx[n], dy[n] = cand[best], ddx[best], ddy[best]
    tt = np.clip(t, 0, None)
    outside = ((lon < BBOX['min_lon'][tt]) | (lon > BBOX['max_lon'][tt]) |
               (lat < BBOX['min_lat'][tt]) | (lat > BBOX['max_lat'][tt]))
    return t, dx, dy, outside

def assign(nt, lat, max_dist):
    """1-based tile, -1 = no tile in pert cell, -2 = dropped. max_dist = ('deg', x) or ('km', x)."""
    t, dx, dy, outside = nt
    units, fov = max_dist
    if units == 'deg':
        mdx, mdy = np.float32(fov), np.float32(fov)
    else:
        mdy = np.float32(fov * KM2DEG)
        mdx = (mdy / np.cos(np.radians(np.asarray(lat, np.float32)))).astype(np.float32)
    too_far = (dx > mdx) | (dy > mdy)
    return np.where(t < 0, -1, np.where(outside & too_far, -2, t + 1))

# ---- tile shapes from the raster window (pixel value = 1-based tile number; land tiles come first)
HAVE_RASTER = TILE_RASTER.exists()
if HAVE_RASTER:
    _r = np.load(TILE_RASTER)
    RAS_TILE, RAS_LAT_E, RAS_LON_E = _r['tile'], _r['lat_e'], _r['lon_e']
    # cube-cell of each land tile, from the tile table (no cell raster needed); non-land pixels -> -1
    CELL_OF_TILE = np.r_[-1, (tc['j_indg'].astype(np.int64) - 1) * 360 + tc['i_indg'].astype(np.int64)]
    print(f'tile raster window: {RAS_TILE.shape}, {np.unique(RAS_TILE).size} tiles '
          f'({RAS_LAT_E[0]:.1f}..{RAS_LAT_E[-1]:.1f}N, {RAS_LON_E[0]:.1f}..{RAS_LON_E[-1]:.1f}E)')
else:
    print(f'no tile raster ({TILE_RASTER}): tiles are drawn as centres of mass')


def raster_window(box):
    # (tile, lat_e, lon_e) cropped to box, or None if the raster does not cover the box
    if not HAVE_RASTER:
        return None
    a, b, c, d = box
    if a < RAS_LAT_E[0] or b > RAS_LAT_E[-1] or c < RAS_LON_E[0] or d > RAS_LON_E[-1]:
        return None
    r0, r1 = np.searchsorted(RAS_LAT_E, a, 'right') - 1, np.searchsorted(RAS_LAT_E, b, 'left')
    c0, c1 = np.searchsorted(RAS_LON_E, c, 'right') - 1, np.searchsorted(RAS_LON_E, d, 'left')
    return RAS_TILE[r0:r1, c0:c1], RAS_LAT_E[r0:r1 + 1], RAS_LON_E[c0:c1 + 1]


def edge_segments(a, lat_e, lon_e):
    # line segments along pixel borders where the id changes (as in the Discover notebook)
    r, c = np.nonzero(a[:, 1:] != a[:, :-1])
    v = np.stack([np.c_[lon_e[c + 1], lat_e[r]], np.c_[lon_e[c + 1], lat_e[r + 1]]], axis=1)
    r, c = np.nonzero(a[1:, :] != a[:-1, :])
    h = np.stack([np.c_[lon_e[c], lat_e[r + 1]], np.c_[lon_e[c + 1], lat_e[r + 1]]], axis=1)
    return np.concatenate([v, h])


def tile_background(ax, win, edge_lw=0.6):
    # land (light grey) / water (light blue), tile edges (light grey) and CF0360 cube-cell edges (dark grey)
    from matplotlib.collections import LineCollection
    tile, lat_e, lon_e = win
    land = tile <= N_LAND
    ax.pcolormesh(lon_e, lat_e, np.where(land, 0, 1), cmap=mcolors.ListedColormap(['0.93', '#cfe3f3']),
                  vmin=-0.5, vmax=1.5, rasterized=True, zorder=0.5)
    cell = np.where(land, CELL_OF_TILE[np.where(land, tile, 0)], -1)
    for ids, color, lw in [(np.where(land, tile, -1), '0.7', edge_lw), (cell, '0.2', 1.4 * edge_lw)]:
        ax.add_collection(LineCollection(edge_segments(ids, lat_e, lon_e), colors=color, lw=lw,
                                         capstyle='projecting', zorder=1.5))


def paint_tiles(ax, win, tiles, values, **kw):
    # colour land tiles (0-based tile index) by value on the raster window; returns the mesh
    tile, lat_e, lon_e = win
    lut = np.full(N_LAND + 1, np.nan)
    lut[np.asarray(tiles) + 1] = values
    land = tile <= N_LAND
    vmap = np.where(land, lut[np.where(land, tile, 0)], np.nan)
    return ax.pcolormesh(lon_e, lat_e, np.ma.masked_invalid(vmap), rasterized=True, zorder=1.0, **kw)


TOL_OLD = ('deg', FOV_OLD_DEG)       # e9c365d: super-ob FOV in deg is also the assignment tolerance
TOL_NEW = ('km', FOV_RAW_KM)         # 9624356 (and raw mode): 12.5 km
print(f'{N_LAND:,} CF0360 land tiles; window ({ANA_TIME - DT_ASSIM / 2:%Y-%m-%d %H:%M}, '
      f'{ANA_TIME + DT_ASSIM / 2:%H:%M}] (+64 s)')

### The four operators

All return sparse weight matrices (one row per obs, one column per land tile, rows sum to 1); `op_tiles` gives
`(tiles, ŵ)` for one obs. The search uses tile centres of mass and the degree-space ellipse of
`get_tile_num_in_ellipse` (no dateline wrap); weights carry the tile area.

- **(R)** raw obs, `FOV = 12.5 km`: ellipse `r_y = 2·12.5 km` in degrees of latitude, `r_x = r_y/cos(lat)`;
  `w = exp(−½·4·nd²)·A` = `exp(−½ (d/12.5 km)²)·A`.
- **(O)** old super-ob, `FOV = 0.141°`: circle of radius 0.141° **in degrees**; `w = A`.
- **(N)** new super-ob: `C = 12.5²·I + S`; search radius `2·√λmax(C)` km; keep `q = dᵀC⁻¹d ≤ 4` (d in km, at the obs
  latitude); `w = exp(−½q)·A`.
- **(X)** exact target: the mean of the (normalised) (R) weight vectors of the super-ob's raw obs, each at its own position.

In [ ]:
def _candidates(lat, lon, rx, ry):
    cand = TREE.query_ball_point(np.c_[lon, lat].astype(np.float64), r=np.maximum(rx, ry))
    obs = np.repeat(np.arange(len(lat)), [len(c) for c in cand])
    tile = np.concatenate(cand).astype(np.int64) if len(obs) else np.zeros(0, np.int64)
    dlon = COM_LON[tile].astype(np.float64) - lon[obs]
    dlat = COM_LAT[tile].astype(np.float64) - lat[obs]
    nd2 = (dlon / rx[obs]) ** 2 + (dlat / ry[obs]) ** 2
    return obs, tile, dlon, dlat, nd2


def _to_matrix(obs, tile, w, n):
    W = sparse.csr_matrix((w, (obs, tile)), shape=(n, N_LAND))
    s = np.asarray(W.sum(1)).ravel()
    return sparse.diags(np.where(s > 0, 1 / np.where(s > 0, s, 1), 0)) @ W


def fov_R(lat, lon, fov_km=FOV_RAW_KM):
    lat, lon = np.asarray(lat, np.float64), np.asarray(lon, np.float64)
    ry = np.full(len(lat), FAC_SEARCH * fov_km * KM2DEG)
    rx = ry / np.cos(np.radians(lat))
    obs, tile, _, _, nd2 = _candidates(lat, lon, rx, ry)
    k = nd2 <= 1.0
    return _to_matrix(obs[k], tile[k], np.exp(-0.5 * FAC_SEARCH ** 2 * nd2[k]) * AREA[tile[k]], len(lat))


def fov_O(lat, lon, fov_deg=FOV_OLD_DEG):
    lat, lon = np.asarray(lat, np.float64), np.asarray(lon, np.float64)
    r = np.full(len(lat), fov_deg)
    obs, tile, _, _, nd2 = _candidates(lat, lon, r, r)
    k = nd2 <= 1.0
    return _to_matrix(obs[k], tile[k], AREA[tile[k]], len(lat))


def footprint_C(S, fov_km=FOV_RAW_KM):
    """C = FOV^2 I + S [km^2]; returns cxx, cyy, cxy, det, lambda_max"""
    cxx, cyy, cxy = fov_km ** 2 + S[:, 0], fov_km ** 2 + S[:, 1], S[:, 2]
    det = cxx * cyy - cxy ** 2
    lmax = 0.5 * (cxx + cyy) + np.sqrt(0.25 * (cxx - cyy) ** 2 + cxy ** 2)
    return cxx, cyy, cxy, det, lmax


def fov_N(lat, lon, S, fov_km=FOV_RAW_KM, return_q=False):
    lat, lon = np.asarray(lat, np.float64), np.asarray(lon, np.float64)
    cxx, cyy, cxy, det, lmax = footprint_C(np.asarray(S, np.float64), fov_km)
    ry = FAC_SEARCH * np.sqrt(lmax) * KM2DEG
    rx = ry / np.cos(np.radians(lat))
    obs, tile, dlon, dlat, nd2 = _candidates(lat, lon, rx, ry)
    k = nd2 <= 1.0
    obs, tile, dlon, dlat = obs[k], tile[k], dlon[k], dlat[k]
    dx = dlon * KM_PER_DEG * np.cos(np.radians(lat[obs]))
    dy = dlat * KM_PER_DEG
    q = (cyy[obs] * dx ** 2 - 2 * cxy[obs] * dx * dy + cxx[obs] * dy ** 2) / det[obs]
    k = q <= FAC_SEARCH ** 2
    W = _to_matrix(obs[k], tile[k], np.exp(-0.5 * q[k]) * AREA[tile[k]], len(lat))
    return (W, obs, tile, q) if return_q else W


def mean_rows(W_rows, groups, n_groups, weights=None):
    """row-mean of W_rows within groups (e.g. raw obs -> super-ob); weights default 1/size"""
    groups = np.asarray(groups)
    cnt = np.bincount(groups, minlength=n_groups)
    w = 1.0 / cnt[groups] if weights is None else weights
    M = sparse.csr_matrix((w, (groups, np.arange(len(groups)))), shape=(n_groups, len(groups)))
    return M @ W_rows


def op_tiles(W, row):
    r = W.getrow(row)
    return r.indices, r.data


def n_eff(W):
    return 1.0 / np.asarray(W.multiply(W).sum(1)).ravel()


def compare(WX, W):
    """per row: overlap sum(min), distance of weighted tile centres [km], tiles, effective tiles"""
    ov = np.asarray(WX.minimum(W).sum(1)).ravel()
    la_x, lo_x = WX @ COM_LAT.astype(np.float64), WX @ COM_LON.astype(np.float64)
    la, lo = W @ COM_LAT.astype(np.float64), W @ COM_LON.astype(np.float64)
    dkm = np.hypot(la - la_x, (lo - lo_x) * np.cos(np.radians(la_x))) * KM_PER_DEG
    return pd.DataFrame({'overlap': ov, 'centres km': dkm, 'tiles': W.getnnz(1), 'eff tiles': n_eff(W)})

## 1. What changed, in one table and one picture

| | raw mode (R) | old super-ob (O), `e9c365d` | new super-ob (N), `9624356` |
|---|---|---|---|
| `FOV`, `FOV_units` | 12.5, `'km'` | s/√π = 0.141°, `'deg'` | 12.5, `'km'` + spread S per super-ob |
| search region | circle of 2·FOV = 25 km | circle of 0.141° **in degrees** (15.7 km N–S, 12.4 km E–W at 37.5°N) | ellipse `q ≤ 4` of C = FOV²·I + S |
| weight | Gaussian, σ = 12.5 km | uniform | Gaussian with covariance C |
| footprint size, full 0.25° cell | (one raw obs) | same disk for every super-ob | σ_eff ≈ 14.8 km (§2) |
| depends on the raw-obs layout | — | no | yes, through S |

The schematic below has no data: one 0.25° cell at 37.5°N with (left) a full 4 × 4 layout of raw obs and (right) raw obs
along a diagonal strip only. Thin blue circles: the 1σ footprint (12.5 km) of each raw obs. Black contours: the exact
mixture (mean of the raw Gaussians), at the levels where a single Gaussian would be at 1σ and 2σ. Green: 1σ and 2σ
ellipses of the new C. Purple: the old disk.

In [ ]:
def ellipse_xy(C2, nsig, n=200):
    """points of the n-sigma ellipse of a 2x2 covariance (km)"""
    ev, evec = np.linalg.eigh(C2)
    t = np.linspace(0, 2 * np.pi, n)
    return (evec @ (nsig * np.sqrt(ev)[:, None] * np.vstack([np.cos(t), np.sin(t)]))).T


LAT_DEMO = 37.5
cx_km = SUPEROB_GRID * KM_PER_DEG * np.cos(np.radians(LAT_DEMO))      # cell width  [km]
cy_km = SUPEROB_GRID * KM_PER_DEG                                    # cell height [km]
g = (np.arange(4) + 0.5) / 4 - 0.5
layouts = {'full cell, 4 x 4 raw obs': np.array([(x * cx_km, y * cy_km) for x in g for y in g]),
           'diagonal strip, 5 raw obs': np.array([(t * cx_km, t * cy_km) for t in np.linspace(-0.4, 0.4, 5)])}

fig, axs = plt.subplots(1, 2, figsize=(15, 7.2), constrained_layout=True)
xx, yy = np.meshgrid(np.linspace(-45, 45, 361), np.linspace(-45, 45, 361))
for ax, (name, P) in zip(axs, layouts.items()):
    P = P - P.mean(0)                                                  # super-ob at the mean position
    mix = np.mean([np.exp(-0.5 * ((xx - x) ** 2 + (yy - y) ** 2) / FOV_RAW_KM ** 2) for x, y in P], axis=0)
    S2 = np.cov(P.T, bias=True)
    C2 = FOV_RAW_KM ** 2 * np.eye(2) + S2
    ax.add_patch(Rectangle((-cx_km / 2 - P[:, 0].mean() * 0, -cy_km / 2), cx_km, cy_km, fill=False, ec='tab:red',
                           lw=1.5, ls='--'))
    t = np.linspace(0, 2 * np.pi, 100)
    for x, y in P:
        ax.plot(x + FOV_RAW_KM * np.cos(t), y + FOV_RAW_KM * np.sin(t), color=OPC['R'], lw=0.5, alpha=0.6)
    ax.plot(P[:, 0], P[:, 1], 'k.', ms=8)
    # contours of the mixture at the levels a single Gaussian of the same peak has at 1 and 2 sigma
    ax.contour(xx, yy, mix / mix.max(), levels=[np.exp(-2), np.exp(-0.5)], colors='k', linewidths=[1.5, 2.5])
    for nsig, lw in [(1, 2.5), (2, 1.5)]:
        e = ellipse_xy(C2, nsig)
        ax.plot(e[:, 0], e[:, 1], color=OPC['N'], lw=lw)
    r_ns, r_ew = FOV_OLD_DEG * KM_PER_DEG, FOV_OLD_DEG * KM_PER_DEG * np.cos(np.radians(LAT_DEMO))
    ax.plot(r_ew * np.cos(t), r_ns * np.sin(t), color=OPC['O'], lw=2.5)
    ax.plot(0, 0, '*', color=OPC['N'], ms=16, mec='k')
    ax.set_aspect('equal'); ax.set_xlim(-45, 45); ax.set_ylim(-45, 45)
    ax.set_xlabel('east [km]'); ax.set_ylabel('north [km]')
    sig = np.sqrt(np.linalg.eigvalsh(C2))
    ax.set_title(f'{name}\nnew C: σ = {sig[1]:.1f} x {sig[0]:.1f} km;  old disk: {r_ew:.1f} x {r_ns:.1f} km (uniform)')
axs[0].legend(handles=[Line2D([], [], color=OPC['R'], lw=1, label='raw-obs footprint, 1σ = 12.5 km'),
                       Line2D([], [], color='k', lw=2.5, label='exact mixture (1σ-, 2σ-equivalent levels)'),
                       Line2D([], [], color=OPC['N'], lw=2.5, label='new: 1σ and 2σ of C = FOV²I + S'),
                       Line2D([], [], color=OPC['O'], lw=2.5, label='old: disk of the cell area'),
                       Line2D([], [], color='tab:red', ls='--', label='0.25° super-ob cell')],
              loc='upper left', fontsize=9)
fig.savefig(FIG_DIR / 'schematic.png', dpi=150)
plt.show()

**Takeaway:** the new footprint is built from the raw obs, so it follows their extent and shape; the old disk is the
same whatever the layout.

## 2. Raw obs, super-obs and their spread S, for the case window

All QC-passing raw obs of the window (all three Metops, global), averaged per satellite into super-obs as in the
reader: cell `i = int((lon+180)/s)+1`, `j = int((lat+90)/s)+1`, position = mean of the raw positions, and the spread S
from the second moments of the offsets from the **cell centre** (1/N), converted to km at the super-ob's mean latitude.
Tile assignment and collisions as in the reader, with the old (0.141°) and the new (12.5 km) tolerance.

In [ ]:
frames, counts = [], []
for sat in SATS:
    for f in window_files(sat):
        df = read_h121(f, sat)
        frames.append(df)
        counts.append({'sat': sat, 'file': f.name.split('_')[-6][:12], 'QC-passing obs': len(df)})
RAW = pd.concat(frames, ignore_index=True)
print(pd.DataFrame(counts).to_string(index=False))
print(f'{len(RAW):,} QC-passing raw obs in the window')


def superobs(raw, grid):
    """super-obs per (sat, j, i) in the reader's order, with S [km^2]; also returns each raw obs' super-ob row"""
    n_lon, n_lat = int(round(360 / grid)), int(round(180 / grid))
    lon = np.mod(raw.lon.to_numpy(np.float64) + 180.0, 360.0) - 180.0
    lat = raw.lat.to_numpy(np.float64)
    i = np.minimum(n_lon, ((lon + 180.0) / grid).astype(int) + 1)
    j = np.minimum(n_lat, ((lat + 90.0) / grid).astype(int) + 1)
    sat_id = raw.sat.map({s: k for k, s in enumerate(SATS)}).to_numpy()
    key = (sat_id.astype(np.int64) * (n_lat + 1) + j) * (n_lon + 1) + i
    uk, inv = np.unique(key, return_inverse=True)                       # sorted: sat, then j outer, i inner
    N = np.bincount(inv)
    dlon = lon - (-180.0 + (i - 0.5) * grid)                           # offsets from the cell centre (double)
    dlat = lat - (-90.0 + (j - 0.5) * grid)
    s = lambda x: np.bincount(inv, x) / N
    mlat, mlon = s(lat), s(lon)
    so = pd.DataFrame({'sat': np.array(SATS)[uk // ((n_lat + 1) * (n_lon + 1))],
                       'j': (uk // (n_lon + 1)) % (n_lat + 1), 'i': uk % (n_lon + 1),
                       'lat': mlat.astype(np.float32), 'lon': mlon.astype(np.float32),
                       'sm': s(raw.sm.to_numpy(np.float64)), 'n_raw': N})
    mdlon, mdlat = s(dlon), s(dlat)
    var_lon = np.maximum(0.0, s(dlon * dlon) - mdlon ** 2)
    var_lat = np.maximum(0.0, s(dlat * dlat) - mdlat ** 2)
    cov = s(dlon * dlat) - mdlon * mdlat
    coslat = np.cos(np.pi / 180.0 * so.lat.to_numpy(np.float64))   # cos of the float32 mean latitude
    so['Sxx'] = (var_lon * (KM_PER_DEG * coslat) ** 2)
    so['Syy'] = (var_lat * KM_PER_DEG ** 2)
    so['Sxy'] = (cov * KM_PER_DEG ** 2 * coslat)
    so['cell_lat0'] = -90 + (so.j - 1) * grid
    so['cell_lon0'] = -180 + (so.i - 1) * grid
    return so, inv


SO, RAW['so'] = superobs(RAW, SUPEROB_GRID)
S = SO[['Sxx', 'Syy', 'Sxy']].to_numpy()


def emitted(so, tiles):
    """one super-ob per (sat, tile): the cell nearest the tile's centre of mass wins (reader order, real*4 dist)"""
    t0 = np.clip(tiles - 1, 0, None)
    dlon = np.abs(so.lon.to_numpy(np.float64) - COM_LON[t0].astype(np.float64))
    dlon = np.minimum(dlon, 360 - dlon)
    d2 = dlon ** 2 + (so.lat.to_numpy(np.float64) - COM_LAT[t0].astype(np.float64)) ** 2
    holder = {}
    for n, (sat, t, dd) in enumerate(zip(so.sat.to_numpy(), tiles, d2)):
        if t <= 0:
            continue
        h = holder.get((sat, t))
        if h is None or dd < np.float64(np.float32(h[1])):
            holder[sat, t] = (n, dd)
    out = np.zeros(len(so), bool)
    out[[n for n, _ in holder.values()]] = True
    return out

NT_SO = nearest_tile(SO.lat, SO.lon)
for tag, tol in [('old', TOL_OLD), ('new', TOL_NEW)]:
    SO[f'tile_{tag}'] = assign(NT_SO, SO.lat, tol)
    SO[f'emitted_{tag}'] = emitted(SO, SO[f'tile_{tag}'].to_numpy())

summary = SO.groupby('sat').agg(**{'occupied cells': ('n_raw', 'size'), 'raw obs': ('n_raw', 'sum'),
                                   'emitted (old tol.)': ('emitted_old', 'sum'),
                                   'emitted (new tol.)': ('emitted_new', 'sum')})
print(summary.to_string())

# raw mode: raw obs -> tiles (12.5 km), averaged per (sat, tile)
NT_RAW = nearest_tile(RAW.lat, RAW.lon)
RAW['tile'] = assign(NT_RAW, RAW.lat, TOL_NEW)

### 2.1 Check: the reader's formula for S equals the population covariance of the raw positions

The reader accumulates offsets from the cell centre (to avoid cancellation); here S is recomputed directly as the
population covariance of the raw lat/lon about their mean, in km at the mean latitude.

In [ ]:
lon_n = np.mod(RAW.lon.to_numpy(np.float64) + 180.0, 360.0) - 180.0
lat_r = RAW.lat.to_numpy(np.float64)
g = RAW.so.to_numpy()
N_ = SO.n_raw.to_numpy()
mlo, mla = np.bincount(g, lon_n) / N_, np.bincount(g, lat_r) / N_
dlo, dla = lon_n - mlo[g], lat_r - mla[g]
coslat = np.cos(np.radians(SO.lat.to_numpy(np.float64)))
S_direct = np.c_[np.bincount(g, dlo * dlo) / N_ * (KM_PER_DEG * coslat) ** 2,
                 np.bincount(g, dla * dla) / N_ * KM_PER_DEG ** 2,
                 np.bincount(g, dlo * dla) / N_ * KM_PER_DEG ** 2 * coslat]
S_DIFF = np.abs(S - S_direct).max()
print(f'max |S(reader formula) - S(direct)| = {S_DIFF:.2e} km^2 over {len(SO):,} super-obs')

### 2.2 How big are the new footprints?

σ_eff = √(12.5² + ½(S_xx + S_yy)), the "mean footprint sigma [km]" that `9624356` writes to the `ldas_log`, over the
emitted super-obs. For reference, a uniformly covered cell has S ≈ (s·k)²/12 per axis: σ_eff ≈ 14.8 km at s = 0.25° and
≈ 20 km at s = 0.5° (N–S, mid-latitudes).

In [ ]:
SO['sigma_eff'] = np.sqrt(FOV_RAW_KM ** 2 + 0.5 * (SO.Sxx + SO.Syy))
SO['n_grp'] = np.where(SO.n_raw >= 6, '6+', SO.n_raw.astype(str))
E = SO[SO.emitted_old]
tab = E.groupby('n_grp').sigma_eff.describe(percentiles=[0.1, 0.5, 0.9])[['count', 'min', '10%', '50%', '90%', 'max']]
print('sigma_eff [km] of emitted super-obs, by number of raw obs')
print(tab.round(2).to_string())
print(f'\nall emitted: mean sigma_eff {E.sigma_eff.mean():.2f} km (the ldas_log line), median {E.sigma_eff.median():.2f} km')
for g_ in [SUPEROB_GRID, 0.5]:
    print(f'uniform full cell, s = {g_}: sigma_eff (N-S) = {np.sqrt(FOV_RAW_KM**2 + (g_ * KM_PER_DEG)**2 / 12):.1f} km')

In [ ]:
def finish(ax, box):
    a, b, c, d = box
    ax.set_xlim(c, d); ax.set_ylim(a, b)
    ax.set_aspect(1 / np.cos(np.radians(0.5 * (a + b))))
    ax.set_xlabel('lon'); ax.set_ylabel('lat')

def inbox(df, box):
    a, b, c, d = box
    return df[df.lat.between(a, b) & df.lon.between(c, d)]

def draw_C(ax, lat, lon, Srow, nsig=1, **kw):
    C2 = FOV_RAW_KM ** 2 * np.eye(2) + np.array([[Srow[0], Srow[2]], [Srow[2], Srow[1]]])
    e = ellipse_xy(C2, nsig)
    ax.plot(lon + e[:, 0] * KM2DEG / np.cos(np.radians(lat)), lat + e[:, 1] * KM2DEG, **kw)

def superob_grid_lines(ax, box, **kw):
    a, b, c, d = box
    for x in np.arange(np.ceil(c / SUPEROB_GRID) * SUPEROB_GRID, d, SUPEROB_GRID):
        ax.axvline(x, **kw)
    for y in np.arange(np.ceil(a / SUPEROB_GRID) * SUPEROB_GRID, b, SUPEROB_GRID):
        ax.axhline(y, **kw)

near_r = np.flatnonzero((COM_LAT > REGION[0]) & (COM_LAT < REGION[1]) & (COM_LON > REGION[2]) & (COM_LON < REGION[3]))
so_r = inbox(SO[(SO.sat == SHOW_SAT) & SO.emitted_old], REGION)
raw_r = inbox(RAW[RAW.sat == SHOW_SAT], REGION)
fig, ax = plt.subplots(figsize=(13, 9), constrained_layout=True)
win_r = raster_window(REGION)
if win_r is not None:
    tile_background(ax, win_r, edge_lw=0.4)
else:
    ax.plot(COM_LON[near_r], COM_LAT[near_r], '.', color='0.82', ms=2)
superob_grid_lines(ax, REGION, color='tab:red', ls='--', lw=0.7)
ax.plot(raw_r.lon, raw_r.lat, '.', color='k', ms=3)
sc = ax.scatter(so_r.lon, so_r.lat, c=so_r.sigma_eff, cmap='viridis', vmin=FOV_RAW_KM, vmax=so_r.sigma_eff.quantile(0.98),
                s=60, edgecolor='k', zorder=4)
for _, r in so_r.iterrows():
    draw_C(ax, r.lat, r.lon, (r.Sxx, r.Syy, r.Sxy), 1, color=plt.get_cmap('viridis')(
        (r.sigma_eff - FOV_RAW_KM) / max(1e-9, so_r.sigma_eff.quantile(0.98) - FOV_RAW_KM)), lw=1)
finish(ax, REGION)
fig.colorbar(sc, ax=ax, shrink=0.7, label='σ_eff = √(12.5² + ½ tr S) [km]')
ax.set_title(f'{SAT_LABEL[SHOW_SAT]}, {ANA_TIME:%Y-%m-%d %H}z: raw obs (black), super-obs coloured by σ_eff, '
             f'1σ ellipses of C' + ('; grey lines = tiles, dark = CF0360 cells' if win_r is not None else '; grey = tile centres of mass'))
fig.savefig(FIG_DIR / 'region_sigma_eff.png', dpi=150)
plt.show()

**Takeaway:** at 0.25° most super-ob footprints are only modestly larger than one raw footprint (σ_eff ≈ 13–15 km);
partly filled cells (swath edges, QC gaps) give elongated, tilted ellipses.

## 3. One super-ob, four footprints

Selection as in the Discover notebook: a super-ob of exactly 3 raw obs, which raw mode keeps all three of, each alone in
its own tile; the one nearest the ZOOM centre. Panels: (X) exact mixture, (N) new, (O) old, (R) the three raw-mode
footprints combined. In this selection every raw-mode obs is a single raw obs at its own position, so (R) equals (X)
here by construction; it differs in general (§4). Tiles are coloured by ŵ (one colour scale for all panels), drawn with
their true shapes from the raster window (light grey lines = tile edges, dark = CF0360 cube cells, light blue = water);
black + = centre of mass of a tile in the footprint, grey + = not in it.

In [ ]:
# raw-mode obs: raw obs averaged per (sat, tile)
RM = (RAW[RAW.tile > 0].groupby(['sat', 'tile'])
      .agg(lat=('lat', lambda x: np.mean(x.astype(np.float64))), lon=('lon', lambda x: np.mean(x.astype(np.float64))),
           n_raw=('lat', 'size')).reset_index())
RM_index = {k: n for n, k in enumerate(zip(RM.sat, RM.tile))}
RAW['rm'] = [RM_index.get(k, -1) if k[1] > 0 else -1 for k in zip(RAW.sat, RAW.tile)]

# operators for all emitted super-obs within REGION + MARGIN (used in sections 3, 4, 5, 7)
box_m = (REGION[0] - MARGIN, REGION[1] + MARGIN, REGION[2] - MARGIN, REGION[3] + MARGIN)
SOR = inbox(SO[SO.emitted_old], box_m).copy()
rows = SOR.index.to_numpy()
raw_sel = RAW[RAW.so.isin(rows)]
grp = pd.Series(np.arange(len(rows)), index=rows)[raw_sel.so].to_numpy()
W_X = mean_rows(fov_R(raw_sel.lat, raw_sel.lon), grp, len(rows))
W_N = fov_N(SOR.lat, SOR.lon, SOR[['Sxx', 'Syy', 'Sxy']].to_numpy())
W_O = fov_O(SOR.lat, SOR.lon)
# (R) mean of the raw-mode footprints the super-ob's raw obs went to (raw obs dropped by raw mode left out)
W_RMall = fov_R(RM.lat, RM.lon)
kept = raw_sel.rm.to_numpy() >= 0
W_RM = mean_rows(W_RMall[raw_sel.rm.to_numpy()[kept]], grp[kept], len(rows))
SOR['n_kept'] = np.bincount(grp[kept], minlength=len(rows))

CENTRE = (np.mean(ZOOM[:2]), np.mean(ZOOM[2:]))
cands = []
for n, (idx, s) in enumerate(SOR.iterrows()):
    if s.n_raw != 3 or not (REGION[0] <= s.lat <= REGION[1] and REGION[2] <= s.lon <= REGION[3]):
        continue
    r = RAW[RAW.so == idx]
    if (r.tile > 0).all() and r.tile.nunique() == 3 and (RM.n_raw.to_numpy()[r.rm.to_numpy()] == 1).all():
        cands.append((np.hypot(s.lat - CENTRE[0], (s.lon - CENTRE[1]) * np.cos(np.radians(s.lat))), n))
CASES = [n for _, n in sorted(cands)]
print(f'{len(CASES)} qualifying super-obs in REGION (all satellites)')

In [ ]:
def weight_panel(ax, box, W, row, vmax, title):
    # true tile shapes where the raster window covers the box, otherwise tile centres of mass sized by area
    near = np.flatnonzero((COM_LAT > box[0]) & (COM_LAT < box[1]) & (COM_LON > box[2]) & (COM_LON < box[3]))
    t, w = op_tiles(W, row)
    inside = np.isin(near, t)
    win = raster_window(box)
    if win is not None:
        tile_background(ax, win)
        sc = paint_tiles(ax, win, t, w, cmap='YlOrRd', vmin=0, vmax=vmax)
        ax.plot(COM_LON[near[~inside]], COM_LAT[near[~inside]], '+', color='0.55', ms=5, mew=0.8, zorder=3)
        ax.plot(COM_LON[near[inside]], COM_LAT[near[inside]], '+', color='k', ms=6, mew=1.2, zorder=3)
    else:
        ax.plot(COM_LON[near[~inside]], COM_LAT[near[~inside]], '+', color='0.6', ms=5, mew=0.8)
        order = np.argsort(w)
        sc = ax.scatter(COM_LON[t[order]], COM_LAT[t[order]], c=w[order], cmap='YlOrRd', vmin=0, vmax=vmax,
                        s=25 + 900 * np.sqrt(AREA[t[order]] / AREA.max()), edgecolor='k', lw=0.3, zorder=3)
    ax.set_title(f'{title}\n{len(t)} tiles, {1 / np.sum(w ** 2):.1f} effective', fontsize=10)
    return sc


def case_figure(n, fig_name):
    s = SOR.iloc[n]
    raws = RAW[RAW.so == SOR.index[n]]
    pad = 0.33
    box = (s.lat - pad, s.lat + pad, s.lon - pad / np.cos(np.radians(s.lat)), s.lon + pad / np.cos(np.radians(s.lat)))
    vmax = max(W.getrow(n).data.max() for W in (W_X, W_N, W_O, W_RM))
    fig, axs = plt.subplots(1, 4, figsize=(21, 6), constrained_layout=True)
    panels = [('X', W_X, '(X) exact: mean of the 3 raw footprints'), ('N', W_N, '(N) new: Gaussian, C = 12.5²I + S'),
              ('O', W_O, '(O) old: 0.141° disk, uniform'), ('R', W_RM, '(R) 3 raw-mode footprints combined')]
    t = np.linspace(0, 2 * np.pi, 120)
    for ax, (k, W, title) in zip(axs, panels):
        sc = weight_panel(ax, box, W, n, vmax, title)
        if k == 'N':
            for nsig, lw in [(1, 2), (2, 1.2)]:
                draw_C(ax, s.lat, s.lon, (s.Sxx, s.Syy, s.Sxy), nsig, color=OPC['N'], lw=lw)
        if k == 'O':
            ax.plot(s.lon + FOV_OLD_DEG * np.cos(t), s.lat + FOV_OLD_DEG * np.sin(t), color=OPC['O'], lw=2)
        if k in ('X', 'R'):
            for _, r in raws.iterrows():
                ry = FAC_SEARCH * FOV_RAW_KM * KM2DEG
                ax.plot(r.lon + ry * np.cos(t) / np.cos(np.radians(r.lat)), r.lat + ry * np.sin(t),
                        color=OPC['R'], lw=0.8)
        ax.add_patch(Rectangle((s.cell_lon0, s.cell_lat0), SUPEROB_GRID, SUPEROB_GRID, fill=False, ec='tab:red',
                               lw=1.2, ls='--'))
        ax.plot(raws.lon, raws.lat, 'k.', ms=9, zorder=5)
        ax.plot(s.lon, s.lat, '*', color='w', ms=16, mec='k', zorder=6)
        finish(ax, box)
    fig.colorbar(sc, ax=axs, shrink=0.8, label='ŵ (normalised tile weight)')
    cmpN, cmpO = compare(W_X[n], W_N[n]).iloc[0], compare(W_X[n], W_O[n]).iloc[0]
    fig.suptitle(f'{SAT_LABEL[s.sat]} super-ob at {s.lat:.2f}N {s.lon:.2f}E ({s.n_raw} raw obs, σ_eff '
                 f'{s.sigma_eff:.1f} km): overlap with X: new {cmpN.overlap:.2f}, old {cmpO.overlap:.2f}; '
                 f'centre shift: new {cmpN["centres km"]:.1f} km, old {cmpO["centres km"]:.1f} km', fontsize=12)
    fig.savefig(FIG_DIR / fig_name, dpi=150)
    plt.show()

if CASES:
    for k, n in enumerate(CASES[:3]):
        case_figure(n, f'four_footprints_case{k + 1}.png')
else:
    print('no qualifying super-ob in REGION for this window; change ANA_TIME or REGION')

Key: black dots = the raw obs; white star = super-ob (mean position); red dashed square = its 0.25° cell; blue circles
= the 25 km search circles of the raw obs; green = 1σ and 2σ ellipses of C; purple = the old 0.141° disk. A tile is in
or out of a footprint as a whole, by its centre of mass. (Without the raster: markers at the centres of mass, size ∝
√(tile area).)

**Takeaway:** (N) is close to the exact footprint (X); (O) is narrower and uniform.

## 4. All super-obs in the region: how close is each operator to the exact one?

For every emitted super-ob in REGION, all satellites, skipping repeats of an earlier satellite's position (the three
Metops often report the same H121 grid points; Discover notebook §4b): overlap Σ min(ŵ_X, ŵ_op) (1 = identical), the
distance between the weighted tile centres, and tile counts. **(R)** is the reference the Discover notebook used (mean of
the raw-mode footprints the raw obs went to; raw obs dropped by raw mode left out); it is close to, but not the same as, (X).

In [ ]:
in_reg = ((SOR.lat >= REGION[0]) & (SOR.lat <= REGION[1]) & (SOR.lon >= REGION[2]) & (SOR.lon <= REGION[3])).to_numpy()
pos = list(zip(SOR.lat.round(4), SOR.lon.round(4)))
seen, repeat = set(), np.zeros(len(SOR), bool)
for n, p in enumerate(pos):                                  # SOR is ordered by satellite (A, B, C)
    repeat[n] = p in seen
    seen.add(p)
U = in_reg & ~repeat & (SOR.n_kept.to_numpy() > 0)
ui = np.flatnonzero(U)
CMP = {k: compare(W_X[ui], W[ui]) for k, W in [('N', W_N), ('O', W_O), ('R', W_RM)]}
cmpX = compare(W_X[ui], W_X[ui])
grp_n = SOR.n_grp.to_numpy()[ui]

rows = []
for gname in ['1', '2', '3', '4', '5', '6+', 'all']:
    m = np.ones(len(ui), bool) if gname == 'all' else grp_n == gname
    if not m.any():
        continue
    r = {'raw obs': gname, 'super-obs': m.sum(), 'X eff tiles': np.median(cmpX['eff tiles'][m])}
    for k in ['N', 'O', 'R']:
        c = CMP[k][m]
        r[f'{k} overlap'] = f"{c.overlap.median():.2f} ({c.overlap.quantile(0.1):.2f}-{c.overlap.quantile(0.9):.2f})"
        r[f'{k} centres km'] = f"{c['centres km'].median():.1f}"
        r[f'{k} eff tiles'] = f"{c['eff tiles'].median():.1f}"
    rows.append(r)
REG_TABLE = pd.DataFrame(rows).set_index('raw obs')
print('medians (10-90th pct) against the exact footprint X')
print(REG_TABLE.to_string())

fig, axs = plt.subplots(1, 2, figsize=(14, 4.6), constrained_layout=True)
for k in ['O', 'N', 'R']:
    axs[0].hist(CMP[k].overlap, bins=np.arange(0, 1.02, 0.02), histtype='step', lw=2, color=OPC[k],
                label=f'{k}: median {CMP[k].overlap.median():.2f}')
    axs[1].hist(CMP[k]['centres km'], bins=np.arange(0, 8, 0.2), histtype='step', lw=2, color=OPC[k], label=k)
axs[0].set_xlabel('overlap Σ min(ŵ_X, ŵ_op)'); axs[0].set_ylabel('super-obs'); axs[0].legend()
axs[1].set_xlabel('distance between weighted tile centres [km]'); axs[1].legend()
fig.suptitle(f'{U.sum()} super-obs in REGION (unique positions, Metop-A/B/C), {ANA_TIME:%Y-%m-%d %H}z', fontsize=12)
fig.savefig(FIG_DIR / 'region_overlap_hist.png', dpi=150)
plt.show()

**Takeaway:** see the overlap medians above: the new footprint shares most of its weight with the exact one, the old disk
much less. Read the numbers in the table for the statement (they depend on the window).

## 5. Partial cells

Super-obs whose raw obs fill only part of the cell (swath edges, QC gaps, coasts) are where the layout matters most.
Below, the three most elongated super-obs (largest √(λmax/λmin) of S; at least 4 raw obs at 3 or more distinct
positions, all kept by raw mode; |lat| < 60°) among **all** emitted super-obs of the window, with (X), (N) and (O).
These lie outside the raster window, so tiles are drawn as centres of mass (marker size ∝ √area).
These are cells where the raw obs lie along a line (S nearly rank-1); C stays well conditioned because of the FOV² term.
At mid-latitudes the next orbit can re-observe the same grid points within the window, so a cell can hold the same
position twice (fewer distinct positions than raw obs).

In [ ]:
ev = np.array([np.linalg.eigvalsh(np.array([[a, c], [c, b]])) for a, b, c in S])
SO['elong'] = np.sqrt(ev[:, 1] / np.maximum(ev[:, 0], 1e-6))
# distinct raw positions per super-ob (at high latitudes the next orbit can re-observe the same grid point
# in the same window, which makes S degenerate); require 3+ distinct positions and |lat| < 60
SO['n_pos'] = RAW.assign(p=list(zip(RAW.lat, RAW.lon))).groupby('so').p.nunique().reindex(SO.index).to_numpy()
cand = SO[SO.emitted_old & (SO.n_raw >= 4) & (SO.n_pos >= 3) & (SO.lat.abs() < 60)].sort_values('elong', ascending=False)
picked = []
for idx, s in cand.iterrows():
    if all(np.hypot(s.lat - SO.lat[p], s.lon - SO.lon[p]) > 2 for p in picked):
        r = RAW[RAW.so == idx]
        if (r.tile > 0).all():
            picked.append(idx)
    if len(picked) == 3:
        break
PS = SO.loc[picked]
rawp = RAW[RAW.so.isin(picked)]
gp = pd.Series(np.arange(len(picked)), index=picked)[rawp.so].to_numpy()
WPX = mean_rows(fov_R(rawp.lat, rawp.lon), gp, len(picked))
WPN = fov_N(PS.lat, PS.lon, PS[['Sxx', 'Syy', 'Sxy']].to_numpy())
WPO = fov_O(PS.lat, PS.lon)

fig, axs = plt.subplots(len(picked), 3, figsize=(16, 5.3 * len(picked)), constrained_layout=True, squeeze=False)
t = np.linspace(0, 2 * np.pi, 120)
for n, (idx, s) in enumerate(PS.iterrows()):
    pad = 0.33
    box = (s.lat - pad, s.lat + pad, s.lon - pad / np.cos(np.radians(s.lat)), s.lon + pad / np.cos(np.radians(s.lat)))
    vmax = max(W.getrow(n).data.max() for W in (WPX, WPN, WPO))
    raws = RAW[RAW.so == idx]
    cN, cO = compare(WPX[n], WPN[n]).iloc[0], compare(WPX[n], WPO[n]).iloc[0]
    for ax, (k, W, title) in zip(axs[n], [('X', WPX, '(X) exact'), ('N', WPN, f'(N) new: overlap {cN.overlap:.2f}'),
                                          ('O', WPO, f'(O) old: overlap {cO.overlap:.2f}')]):
        sc = weight_panel(ax, box, W, n, vmax, f'{title}')
        if k == 'N':
            for nsig, lw in [(1, 2), (2, 1.2)]:
                draw_C(ax, s.lat, s.lon, (s.Sxx, s.Syy, s.Sxy), nsig, color=OPC['N'], lw=lw)
        if k == 'O':
            ax.plot(s.lon + FOV_OLD_DEG * np.cos(t), s.lat + FOV_OLD_DEG * np.sin(t), color=OPC['O'], lw=2)
        ax.add_patch(Rectangle((s.cell_lon0, s.cell_lat0), SUPEROB_GRID, SUPEROB_GRID, fill=False, ec='tab:red', lw=1.2,
                               ls='--'))
        ax.plot(raws.lon, raws.lat, 'k.', ms=9, zorder=5)
        ax.plot(s.lon, s.lat, '*', color='w', ms=16, mec='k', zorder=6)
        finish(ax, box)
    sd = np.sqrt(np.clip(np.linalg.eigvalsh(np.array([[s.Sxx, s.Sxy], [s.Sxy, s.Syy]])), 0, None))
    axs[n, 0].set_ylabel(f'{SAT_LABEL[s.sat]} {s.lat:.2f}N {s.lon:.2f}E\n{s.n_raw} raw obs at {s.n_pos} positions; '
                         f'S axes {sd[1]:.1f} x {sd[0]:.1f} km\nlat')
fig.savefig(FIG_DIR / 'partial_cells.png', dpi=150)
plt.show()

### 5.1 Super-obs lost by the tighter tile-assignment tolerance

`9624356` also uses FOV as the maximum obs-to-tile distance in `get_tile_num_for_obs`; for super-obs this changes from
0.141° to 12.5 km. A super-ob that lies outside its nearest tile's bounding box and is more than 12.5 km (but less than
0.141°) from it in lon or lat now gets no tile.

In [ ]:
lost = SO.emitted_old & (SO.tile_new <= 0)
gained = SO.emitted_new & (SO.tile_old <= 0)
print(f'emitted super-obs: old tolerance {SO.emitted_old.sum():,}, new tolerance {SO.emitted_new.sum():,}')
print(f'lost with the new tolerance: {lost.sum():,} ({lost.sum() / SO.emitted_old.sum():.2%}); '
      f'gained: {gained.sum():,}; changes from tile collisions: '
      f'{(SO.emitted_old != SO.emitted_new).sum() - lost.sum() - gained.sum():,}')
print('lost, by number of raw obs:', SO[lost].n_grp.value_counts().sort_index().to_dict())

fig, ax = plt.subplots(figsize=(12, 5.5), constrained_layout=True)
e = SO[SO.emitted_old]
ax.plot(e.lon, e.lat, ',', color='0.75')
ax.plot(SO.lon[lost], SO.lat[lost], 'o', color='tab:red', ms=3)
ax.set_xlim(-180, 180); ax.set_ylim(-60, 85); ax.set_aspect(1.3)
ax.set_title(f'emitted super-obs (grey) and those lost with the 12.5 km tolerance (red), {ANA_TIME:%Y-%m-%d %H}z')
fig.savefig(FIG_DIR / 'lost_superobs.png', dpi=150)
plt.show()

**Takeaway:** (N) follows the actual layout (an elongated, tilted C) while (O) is the same disk whatever the layout; the
tighter assignment tolerance drops a small fraction of super-obs, mostly at coasts.

## 6. Predicted values

Apply each operator to the same tile field x and compare with the exact prediction H_X x: Δ_N = H_N x − H_X x and
Δ_O = H_O x − H_X x, for **all** emitted super-obs of the window.

**Tile field:** SMAP L4 `SFMC_FCST` (2019-11-01 00z, 9 km EASE tiles), assigned to each CF0360 tile from the nearest
L4 tile within 15 km (other tiles undefined; super-obs with an undefined tile in any footprint are left out). The
date differs from the obs window; that is fine here because Δ measures only how the operators differ on a realistic
soil-moisture pattern. The GEOSldas `SFMC_FCST` of the runs (Discover) would give the same kind of numbers.

(R), the raw-mode reference, is left out for super-obs whose raw obs were all dropped by raw mode (no raw-mode footprint).

Obs-error units: errstd = 9 % saturation × porosity 0.45 ≈ 0.04 m³/m³ (approximate; the scaled obs-error std in
ObsFcstAna varies by location).

In [ ]:
with nc.Dataset(L4_SFMC) as d:
    l4 = pd.DataFrame({'lat': np.asarray(d['lat'][:], np.float64), 'lon': np.asarray(d['lon'][:], np.float64),
                       'sfmc': np.ma.filled(d['SFMC_FCST'][0, :].astype(np.float64), np.nan)})
l4 = l4[np.isfinite(l4.sfmc) & (l4.sfmc >= 0) & (l4.sfmc < 1)]
def xyz(lat, lon):
    la, lo = np.radians(lat), np.radians(lon)
    return np.c_[np.cos(la) * np.cos(lo), np.cos(la) * np.sin(lo), np.sin(la)]
dd, ii = cKDTree(xyz(l4.lat.to_numpy(), l4.lon.to_numpy())).query(xyz(COM_LAT.astype(float), COM_LON.astype(float)))
FIELD = np.where(dd * MAPL_RADIUS_KM < 15.0, l4.sfmc.to_numpy()[ii], np.nan)
print(f'tile field: {np.isfinite(FIELD).mean():.1%} of CF0360 land tiles defined; '
      f'median {np.nanmedian(FIELD):.3f} m3/m3')

# all emitted super-obs (global)
GE = SO[SO.emitted_old].copy()
gi = GE.index.to_numpy()
raw_g = RAW[RAW.so.isin(gi)]
gg = pd.Series(np.arange(len(gi)), index=gi)[raw_g.so].to_numpy()
WG = {'X': mean_rows(fov_R(raw_g.lat, raw_g.lon), gg, len(gi)),
      'N': fov_N(GE.lat, GE.lon, GE[['Sxx', 'Syy', 'Sxy']].to_numpy()),
      'O': fov_O(GE.lat, GE.lon)}
kept_g = raw_g.rm.to_numpy() >= 0
WG['R'] = mean_rows(W_RMall[raw_g.rm.to_numpy()[kept_g]], gg[kept_g], len(gi))

f0 = np.nan_to_num(FIELD)
undef = sparse.diags(1.0 * ~np.isfinite(FIELD))
for k, W in WG.items():
    GE[f'H{k}'] = W @ f0
    GE[f'undef{k}'] = np.asarray((W @ undef).sum(1)).ravel() > 0
ok = ~(GE.undefX | GE.undefN | GE.undefO) & (np.asarray(WG['X'].sum(1)).ravel() > 0) & \
     (np.asarray(WG['N'].sum(1)).ravel() > 0) & (np.asarray(WG['O'].sum(1)).ravel() > 0)
G6 = GE[ok].copy()
G6['dN'], G6['dO'] = G6.HN - G6.HX, G6.HO - G6.HX
# (R) only where at least one raw obs was kept by raw mode (otherwise there is no raw-mode footprint)
G6['has_R'] = (np.asarray(WG['R'].sum(1)).ravel() > 0)[ok.to_numpy()]
G6['dR'] = np.where(G6.undefR | ~G6.has_R, np.nan, G6.HR - G6.HX)
OBSERR = ERRSTD_PCT / 100 * POROSITY

def stats(d):
    d = d.dropna()
    return pd.Series({'n': len(d), 'rms': np.sqrt(np.mean(d ** 2)), 'bias': d.mean(),
                      '|Δ| p90': d.abs().quantile(0.9), '|Δ| p99': d.abs().quantile(0.99),
                      'rms / obs err': np.sqrt(np.mean(d ** 2)) / OBSERR})

rows = []
for gname in ['all', '1', '2', '3', '4', '5', '6+']:
    m = np.ones(len(G6), bool) if gname == 'all' else (G6.n_grp == gname).to_numpy()
    for k in ['N', 'O', 'R']:
        rows.append(pd.concat([pd.Series({'raw obs': gname, 'op': k}), stats(G6[f'd{k}'][m])]))
PRED = pd.DataFrame(rows).set_index(['raw obs', 'op'])
print(f'{len(G6):,} super-obs (of {len(GE):,} emitted) with a defined field in all footprints; Δ in m3/m3')
print(PRED.to_string(float_format=lambda v: f'{v:.4f}'))

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(19, 6), constrained_layout=True)
lim = (0, 0.55)
for ax, k in zip(axs[:2], ['O', 'N']):
    ax.hist2d(G6.HX, G6[f'H{k}'], bins=np.linspace(*lim, 221), cmin=1, cmap='viridis', norm=mcolors.LogNorm(),
              rasterized=True)
    ax.plot(lim, lim, 'r-', lw=0.8)
    ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect('equal')
    d = G6[f'd{k}']
    ax.set_xlabel('H_X x (exact) [m3/m3]'); ax.set_ylabel(f'H_{k} x [m3/m3]')
    ax.set_title(f'({k}) {"old" if k == "O" else "new"}: rms Δ {np.sqrt(np.mean(d**2)):.4f}, bias {d.mean():+.4f}')
bins = np.linspace(-0.06, 0.06, 121)
for k in ['O', 'N']:
    axs[2].hist(G6[f'd{k}'], bins=bins, histtype='step', lw=2, color=OPC[k], label=f'Δ_{k}')
axs[2].axvspan(-OBSERR, OBSERR, color='0.9', zorder=0, label=f'± obs err ≈ {OBSERR:.3f}')
axs[2].set_yscale('log'); axs[2].set_xlabel('Δ = H_op x − H_X x [m3/m3]'); axs[2].legend()
fig.suptitle(f'All {len(G6):,} super-obs, {ANA_TIME:%Y-%m-%d %H}z, Metop-A/B/C; tile field = SMAP L4 SFMC on CF0360 tiles',
             fontsize=12)
fig.savefig(FIG_DIR / 'predicted_values.png', dpi=150)
plt.show()

**Not checked here (Discover-only):** that (O) and (R) reproduce ObsFcstAna `fcst` (done in the Discover notebook §6c,
≤ 5e-5 m³/m³, with the same operator code), and that (N) reproduces the `fcst` of a `9624356` run (pending the retest).

**Takeaway:** compare the rms of Δ_N and Δ_O above: the footprint error of the prediction before vs after the change, in
m³/m³ and in obs-error units.

## 7. Obs spacing and footprint overlap

Super-ob positions do not change, so their spacing, the obs-error matrix R and its conditioning do not change either.
The footprints do: below, (a) the cosine similarity of the tile-weight vectors of two super-obs of the same satellite
against their distance (Discover notebook §7c), for (O) and (N); (b) the fraction of land tiles in REGION (0.3° away from
its edges) that are in no footprint (all satellites).

In [ ]:
def pair_cos(W, sel, max_km=60.0):
    # all same-satellite pairs closer than max_km, with the cosine similarity of their weight vectors (0 if disjoint)
    la, lo, sat = SOR.lat.to_numpy()[sel], SOR.lon.to_numpy()[sel], SOR.sat.to_numpy()[sel]
    Wn = W[sel]
    norms = np.sqrt(np.asarray(Wn.multiply(Wn).sum(1)).ravel())
    G_ = (Wn @ Wn.T).todok()
    xy = np.c_[lo * np.cos(np.radians(la)), la] * KM_PER_DEG
    rows = []
    for a, b in cKDTree(xy).query_pairs(max_km):
        if sat[a] == sat[b]:
            rows.append((np.hypot(*(xy[a] - xy[b])), G_.get((a, b), 0.0) / (norms[a] * norms[b])))
    return np.array(rows)

sel = np.flatnonzero(in_reg)
fig, axs = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
UNUSED = {}
inner = np.flatnonzero((COM_LAT > REGION[0] + 0.3) & (COM_LAT < REGION[1] - 0.3) &
                       (COM_LON > REGION[2] + 0.3) & (COM_LON < REGION[3] - 0.3))
for k, W in [('O', W_O), ('N', W_N)]:
    p = pair_cos(W, sel)
    axs[0].plot(p[:, 0], p[:, 1], '.', ms=4, alpha=0.6, color=OPC[k],
                label=f'{k}: pairs < 30 km median {np.median(p[p[:, 0] < 30, 1]):.2f}')
    used = np.asarray((W[sel] > 0).sum(0)).ravel()
    UNUSED[k] = np.mean(used[inner] == 0)
    axs[1].hist(used[inner], bins=np.arange(-0.5, 25.5), histtype='step', lw=2, color=OPC[k],
                label=f'{k}: {UNUSED[k]:.0%} of tiles unused')
axs[0].set_xlabel('distance between two super-obs of the same satellite [km]'); axs[0].set_ylabel('cosine similarity')
axs[0].set_xlim(0, 60); axs[0].legend(); axs[0].set_title('(a) footprint similarity')
axs[1].set_xlabel('footprints containing a tile (all satellites)'); axs[1].set_ylabel('tiles')
axs[1].legend(); axs[1].set_title('(b) how often a tile is used')
fig.savefig(FIG_DIR / 'overlap_and_coverage.png', dpi=150)
plt.show()

### 7.1 All footprints of one satellite in the zoom box

As in the Discover notebook §6b: every super-ob footprint of one satellite, tiles coloured by **how many footprints
contain them**, for the old and the new operator.

In [ ]:
zbox = ZOOM
win_z = raster_window(zbox)
ext = (zbox[0] - 0.3, zbox[1] + 0.3, zbox[2] - 0.4, zbox[3] + 0.4)
rows_z = np.flatnonzero((SOR.sat == SHOW_SAT).to_numpy() & SOR.lat.between(ext[0], ext[1]).to_numpy()
                        & SOR.lon.between(ext[2], ext[3]).to_numpy())
fig, axs = plt.subplots(1, 2, figsize=(17, 8), constrained_layout=True)
cnorm = mcolors.BoundaryNorm(np.arange(0.5, 9.5), 256)
t = np.linspace(0, 2 * np.pi, 120)
for ax, (k, W) in zip(axs, [('O', W_O), ('N', W_N)]):
    cnt = np.asarray((W[rows_z] > 0).sum(0)).ravel()
    used = np.flatnonzero(cnt)
    if win_z is not None:
        tile_background(ax, win_z)
        pm = paint_tiles(ax, win_z, used, cnt[used].astype(float), cmap='magma_r', norm=cnorm)
    else:
        pm = ax.scatter(COM_LON[used], COM_LAT[used], c=cnt[used], cmap='magma_r', norm=cnorm, s=40)
    for n in rows_z:
        s_ = SOR.iloc[n]
        if k == 'O':
            ax.plot(s_.lon + FOV_OLD_DEG * np.cos(t), s_.lat + FOV_OLD_DEG * np.sin(t), color=OPC['O'], lw=0.9)
        else:
            draw_C(ax, s_.lat, s_.lon, (s_.Sxx, s_.Syy, s_.Sxy), 2, color=OPC['N'], lw=0.9)
    so_z = SOR.iloc[rows_z]
    ax.plot(so_z.lon, so_z.lat, 'o', color=OPC[k], ms=6, mec='k', mew=0.4, zorder=4)
    superob_grid_lines(ax, zbox, color='tab:red', ls='--', lw=0.8)
    inner = np.flatnonzero((COM_LAT > zbox[0]) & (COM_LAT < zbox[1]) & (COM_LON > zbox[2]) & (COM_LON < zbox[3]))
    ax.set_title(f'{"old 0.141° disk" if k == "O" else "new Gaussian (2σ ellipses)"}: '
                 f'{np.mean(cnt[inner] == 0):.0%} of tiles in the box unused, '
                 f'median {np.median(cnt[inner][cnt[inner] > 0]):.0f} footprints per used tile')
    finish(ax, zbox)
fig.colorbar(pm, ax=axs, shrink=0.75, ticks=np.arange(1, 9), label=f'footprints containing the tile ({SAT_LABEL[SHOW_SAT]})')
fig.suptitle(f'{SAT_LABEL[SHOW_SAT]}, {ANA_TIME:%Y-%m-%d %H}z: super-ob footprints and how often each tile is used', fontsize=13)
fig.savefig(FIG_DIR / 'footprint_coverage_zoom.png', dpi=150)
plt.show()

**Takeaway:** the wider Gaussian footprints leave fewer tiles unused but make neighbouring super-obs' footprints more
alike (higher cosine similarity at the typical ~20 km spacing); the obs positions, spacing and R are unchanged.

## 8. Summary

In [ ]:
c_all = {k: CMP[k] for k in CMP}
pa = PRED.loc['all']
lines = [
    f'Case: {ANA_TIME:%Y-%m-%d %H}z window (local H121, Metop-A/B/C), {SUPEROB_GRID}° super-obs on CF0360; '
    f'{SO.emitted_old.sum():,} emitted super-obs globally.',
    f'S from the reader formula = direct covariance (max diff {S_DIFF:.1e} km²).',
    f'New footprint size: mean σ_eff {E.sigma_eff.mean():.1f} km (median {E.sigma_eff.median():.1f}; 12.5 km for '
    f'single-obs cells; the old disk is 0.141° ≈ 15.7 x 12.4 km, uniform).',
    f'Overlap with the exact footprint X in REGION ({len(ui)} super-obs): new median {c_all["N"].overlap.median():.2f}, '
    f'old {c_all["O"].overlap.median():.2f}, raw-mode reference {c_all["R"].overlap.median():.2f}; '
    f'weighted-centre shift median new {c_all["N"]["centres km"].median():.1f} km, old {c_all["O"]["centres km"].median():.1f} km.',
    f'Prediction error vs H_X (all super-obs, SMAP L4 field): rms new {pa.loc["N", "rms"]:.4f}, old '
    f'{pa.loc["O", "rms"]:.4f} m3/m3 ({pa.loc["N", "rms / obs err"]:.2f} vs {pa.loc["O", "rms / obs err"]:.2f} obs-error std).',
    f'Tighter assignment tolerance (12.5 km): {lost.sum():,} super-obs lost ({lost.sum() / SO.emitted_old.sum():.2%}).',
    f'Tiles in REGION used by no footprint: old {UNUSED["O"]:.0%}, new {UNUSED["N"]:.0%}.',
]
print('\n'.join('- ' + s for s in lines))

Settings to use: keep `FOV = 12.5`, `FOV_units = 'km'` for the H SAF species and set `superob_grid_deg` as before; the
super-ob footprint is derived from FOV and each super-ob's spread. The old `'deg'` namelist now aborts at startup.

## 9. Two figures that capture the change

For presentations: the super-ob of §3 (case 1).

1. **Footprints on the tiles:** what the three raw obs would see as three raw-mode obs (their footprints combined), and
   what the one super-ob sees with the new Gaussian.
2. **How the Gaussians are combined** (continuous, no tiles): each raw obs has a 12.5 km Gaussian footprint; the super-ob
   is their mean, so its exact footprint is the mean of the Gaussians (a mixture); GEOSldas replaces the mixture by the
   single Gaussian with the same mean and covariance, C = 12.5²·I + S, where S is the spread of the raw obs.

In [ ]:
from matplotlib.patches import Patch

n0 = CASES[0]
s0 = SOR.iloc[n0]
raws0 = RAW[RAW.so == SOR.index[n0]]
pad = 0.33
box0 = (s0.lat - pad, s0.lat + pad, s0.lon - pad / np.cos(np.radians(s0.lat)), s0.lon + pad / np.cos(np.radians(s0.lat)))
vmax0 = max(W.getrow(n0).data.max() for W in (W_RM, W_N))
tt = np.linspace(0, 2 * np.pi, 120)

fig, axs = plt.subplots(1, 2, figsize=(13, 6.6), constrained_layout=True)
weight_panel(axs[0], box0, W_RM, n0, vmax0, f'3 raw obs as 3 raw-mode obs: footprints combined')
sc = weight_panel(axs[1], box0, W_N, n0, vmax0, f'1 super-ob, new footprint: Gaussian, C = 12.5²I + S')
for _, r in raws0.iterrows():
    ry = FAC_SEARCH * FOV_RAW_KM * KM2DEG
    axs[0].plot(r.lon + ry * np.cos(tt) / np.cos(np.radians(r.lat)), r.lat + ry * np.sin(tt), color=OPC['R'], lw=1.2)
for nsig, lw in [(1, 2.2), (2, 1.3)]:
    draw_C(axs[1], s0.lat, s0.lon, (s0.Sxx, s0.Syy, s0.Sxy), nsig, color=OPC['N'], lw=lw)
for ax in axs:
    ax.add_patch(Rectangle((s0.cell_lon0, s0.cell_lat0), SUPEROB_GRID, SUPEROB_GRID, fill=False, ec='tab:red', lw=1.3, ls='--'))
    ax.plot(raws0.lon, raws0.lat, 'k.', ms=10, zorder=5)
    finish(ax, box0)
axs[1].plot(s0.lon, s0.lat, '*', color='w', ms=17, mec='k', zorder=6)
axs[1].set_ylabel('')
ov = compare(W_RM[n0], W_N[n0]).iloc[0]
fig.colorbar(sc, ax=axs, shrink=0.75, label='tile weight (each panel sums to 1)')
fig.legend(handles=[Line2D([], [], marker='.', color='k', ls='none', ms=10, label='raw H121 obs'),
                    Line2D([], [], color=OPC['R'], lw=1.2, label='raw-obs search circle (25 km = 2 x 12.5 km)'),
                    Line2D([], [], marker='*', color='w', mec='k', ls='none', ms=13, label='super-ob (mean position)'),
                    Line2D([], [], color=OPC['N'], lw=2.2, label='1σ and 2σ of C'),
                    Line2D([], [], color='tab:red', ls='--', label=f'{SUPEROB_GRID}° super-ob cell')]
           + ([Patch(fc='0.93', ec='0.7', label='tiles (dark lines: CF0360 cells)')] if raster_window(box0) is not None else []),
           loc='outside lower center', ncol=3, fontsize=9, frameon=False)
fig.suptitle(f'{SAT_LABEL[s0.sat]}, {s0.lat:.2f}N {s0.lon:.2f}E: weight overlap {ov.overlap:.2f}, '
             f'weighted centres {ov["centres km"]:.1f} km apart', fontsize=12)
fig.savefig(FIG_DIR / 'summary_footprints_rawmode_vs_new.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# the same super-ob in km around its mean position
P = np.c_[(raws0.lon.to_numpy(np.float64) - s0.lon) * KM_PER_DEG * np.cos(np.radians(s0.lat)),
          (raws0.lat.to_numpy(np.float64) - s0.lat) * KM_PER_DEG]
S2 = np.array([[s0.Sxx, s0.Sxy], [s0.Sxy, s0.Syy]])
C2 = FOV_RAW_KM ** 2 * np.eye(2) + S2
L = 42
xx, yy = np.meshgrid(np.linspace(-L, L, 421), np.linspace(-L, L, 421))

def gauss(C, mx=0.0, my=0.0):
    Ci = np.linalg.inv(C)
    dx, dy = xx - mx, yy - my
    return np.exp(-0.5 * (Ci[0, 0] * dx**2 + 2 * Ci[0, 1] * dx * dy + Ci[1, 1] * dy**2)) / (2 * np.pi * np.sqrt(np.linalg.det(C)))

G_raw = [gauss(FOV_RAW_KM ** 2 * np.eye(2), x, y) for x, y in P]
MIX = np.mean(G_raw, axis=0)
NEW = gauss(C2)
vmax = max(MIX.max(), NEW.max())
cols = ['tab:blue', 'tab:orange', 'tab:purple', 'tab:brown', 'tab:pink', 'tab:olive']

fig, axs = plt.subplots(1, 3, figsize=(18, 6.4), constrained_layout=True)
ax = axs[0]
for k, (g, (x, y)) in enumerate(zip(G_raw, P)):
    c = cols[k % len(cols)]
    ax.contourf(xx, yy, g, levels=[g.max() * np.exp(-2), g.max() * np.exp(-0.5), g.max()], colors=[c, c], alpha=0.12)
    for nsig, lw in [(1, 1.8), (2, 1.0)]:
        ax.plot(x + nsig * FOV_RAW_KM * np.cos(tt), y + nsig * FOV_RAW_KM * np.sin(tt), color=c, lw=lw)
    ax.plot(x, y, 'o', color=c, mec='k', ms=8, zorder=5)
ax.set_title(f'(a) each raw obs k: Gaussian footprint\nN(x_k, 12.5² I)  (circles: 1σ, 2σ)')

ax = axs[1]
ax.pcolormesh(xx, yy, MIX, cmap='Greys', vmin=0, vmax=vmax, rasterized=True)
ax.contour(xx, yy, MIX, levels=MIX.max() * np.array([np.exp(-2), np.exp(-0.5)]), colors='k', linewidths=[1, 1.8])
eS = ellipse_xy(S2 + 1e-9 * np.eye(2), 1)
ax.plot(eS[:, 0], eS[:, 1], 'r--', lw=1.5)
ax.plot(P[:, 0], P[:, 1], 'o', color='w', mec='k', ms=8, zorder=5)
ax.plot(0, 0, '*', color='w', mec='k', ms=17, zorder=6)
ax.set_title(f'(b) super-ob = mean of the {len(P)} raw obs\nexact footprint = (1/N) Σ_k N(x_k, 12.5² I)  (red: spread S)')

ax = axs[2]
ax.pcolormesh(xx, yy, NEW, cmap='Greens', vmin=0, vmax=vmax, rasterized=True)
ax.contour(xx, yy, MIX, levels=MIX.max() * np.array([np.exp(-2), np.exp(-0.5)]), colors='k', linewidths=[1, 1.8],
           linestyles='--')
for nsig, lw in [(1, 2.2), (2, 1.3)]:
    e = ellipse_xy(C2, nsig)
    ax.plot(e[:, 0], e[:, 1], color='darkgreen', lw=lw)
ax.plot(P[:, 0], P[:, 1], 'o', color='w', mec='k', ms=8, zorder=5)
ax.plot(0, 0, '*', color='w', mec='k', ms=17, zorder=6)
sig = np.sqrt(np.linalg.eigvalsh(C2))
ax.set_title(f'(c) new footprint: one Gaussian, same mean and covariance\nC = 12.5² I + S  (σ = {sig[1]:.1f} x {sig[0]:.1f} km; '
             f'dashed: mixture from b)')
for ax in axs:
    ax.set_aspect('equal'); ax.set_xlim(-L, L); ax.set_ylim(-L, L)
    ax.set_xlabel('east [km]')
axs[0].set_ylabel('north [km]')
fig.savefig(FIG_DIR / 'summary_gaussian_combination.png', dpi=200, bbox_inches='tight')
plt.show()

### 9.1 The same cartoon for super-obs of one, two, three and four raw obs

Real super-obs from the Kansas box (each made of 1, 2, 3 or 4 raw obs at distinct positions; the 3-obs row is the
super-ob above). Columns as above: (a) the raw-obs footprints, (b) their mean, the exact footprint, (c) the new single
Gaussian with the mixture dashed. The number in (c) is the overlap ∫ min(mixture, Gaussian) (1 = identical).
With one raw obs S = 0, so the new footprint *is* the raw-obs footprint.

In [ ]:
def picked_superob(k):
    if k == 3:
        return CASES[0]
    npos = SO.loc[SOR.index, 'n_pos'].to_numpy()
    cand = np.flatnonzero(in_reg & (SOR.n_raw.to_numpy() == k) & (npos == k))
    d = np.hypot(SOR.lat.to_numpy()[cand] - CENTRE[0],
                 (SOR.lon.to_numpy()[cand] - CENTRE[1]) * np.cos(np.radians(CENTRE[0])))
    return cand[np.argmin(d)]


def superob_km(n):
    # raw positions [km] about the super-ob's mean position, and its spread S [km^2]
    s = SOR.iloc[n]
    r = RAW[RAW.so == SOR.index[n]]
    P_ = np.c_[(r.lon.to_numpy(np.float64) - s.lon) * KM_PER_DEG * np.cos(np.radians(s.lat)),
               (r.lat.to_numpy(np.float64) - s.lat) * KM_PER_DEG]
    return P_, np.array([[s.Sxx, s.Sxy], [s.Sxy, s.Syy]])


DA = (xx[0, 1] - xx[0, 0]) ** 2                       # grid cell area [km^2]

def cartoon_row(axs, P_, S_):
    C_ = FOV_RAW_KM ** 2 * np.eye(2) + S_
    G_ = [gauss(FOV_RAW_KM ** 2 * np.eye(2), x, y) for x, y in P_]
    mix, new = np.mean(G_, axis=0), gauss(C_)
    vmx = max(mix.max(), new.max())
    lev = lambda a: a.max() * np.array([np.exp(-2), np.exp(-0.5)])
    ax = axs[0]
    for k, (g, (x, y)) in enumerate(zip(G_, P_)):
        c = cols[k % len(cols)]
        ax.contourf(xx, yy, g, levels=[g.max() * np.exp(-2), g.max() * np.exp(-0.5), g.max()], colors=[c, c], alpha=0.12)
        for nsig, lw in [(1, 1.8), (2, 1.0)]:
            ax.plot(x + nsig * FOV_RAW_KM * np.cos(tt), y + nsig * FOV_RAW_KM * np.sin(tt), color=c, lw=lw)
        ax.plot(x, y, 'o', color=c, mec='k', ms=8, zorder=5)
    ax = axs[1]
    ax.pcolormesh(xx, yy, mix, cmap='Greys', vmin=0, vmax=vmx, rasterized=True)
    ax.contour(xx, yy, mix, levels=lev(mix), colors='k', linewidths=[1, 1.8])
    if np.trace(S_) > 0:
        eS = ellipse_xy(S_ + 1e-9 * np.eye(2), 1)
        ax.plot(eS[:, 0], eS[:, 1], 'r--', lw=1.5)
    ax = axs[2]
    ax.pcolormesh(xx, yy, new, cmap='Greens', vmin=0, vmax=vmx, rasterized=True)
    ax.contour(xx, yy, mix, levels=lev(mix), colors='k', linewidths=[1, 1.8], linestyles='--')
    for nsig, lw in [(1, 2.2), (2, 1.3)]:
        e = ellipse_xy(C_, nsig)
        ax.plot(e[:, 0], e[:, 1], color='darkgreen', lw=lw)
    for ax in axs[1:]:
        ax.plot(P_[:, 0], P_[:, 1], 'o', color='w', mec='k', ms=8, zorder=5)
        ax.plot(0, 0, '*', color='w', mec='k', ms=17, zorder=6)
    for ax in axs:
        ax.set_aspect('equal'); ax.set_xlim(-L, L); ax.set_ylim(-L, L)
    sig = np.sqrt(np.linalg.eigvalsh(C_))
    return sig, np.minimum(mix, new).sum() / mix.sum()        # overlap of the parts inside the plotted area


fig, axs = plt.subplots(4, 3, figsize=(16.5, 22), constrained_layout=True)
for row, k in enumerate([1, 2, 3, 4]):
    n = picked_superob(k)
    P_, S_ = superob_km(n)
    sig, ov = cartoon_row(axs[row], P_, S_)
    s = SOR.iloc[n]
    axs[row, 0].set_ylabel(f'{k} raw obs\n({SAT_LABEL[s.sat]}, {s.lat:.2f}N {s.lon:.2f}E)\nnorth [km]', fontsize=11)
    axs[row, 0].set_title('(a) raw-obs footprints N(x_k, 12.5² I)' if row == 0 else '')
    axs[row, 1].set_title('(b) exact: (1/N) Σ_k N(x_k, 12.5² I)  (red: S)' if row == 0 else '')
    axs[row, 2].set_title(('(c) new: N(μ, 12.5² I + S)\n' if row == 0 else '')
                          + f'σ = {sig[1]:.1f} x {sig[0]:.1f} km; overlap with (b) {ov:.3f}')
for ax in axs[-1]:
    ax.set_xlabel('east [km]')
fig.savefig(FIG_DIR / 'summary_gaussian_combination_1to4.png', dpi=200, bbox_inches='tight')
plt.show()

### 9.2 The math behind the new footprint

**1. One raw obs.** A raw obs $k$ at position $\mathbf{x}_k$ is predicted from the tile values $x_t$ as
$H_k x = \sum_t \hat w_{k,t}\, x_t$, with tile weights

$$\hat w_{k,t} \propto A_t \, g_k(\mathbf{c}_t), \qquad g_k(\mathbf{r}) = \exp\!\left(-\frac{|\mathbf{r}-\mathbf{x}_k|^2}{2\sigma^2}\right), \qquad \sigma = \text{FOV} = 12.5\ \text{km},$$

where $A_t$ is the tile area and $\mathbf{c}_t$ its centre of mass; tiles beyond $2\sigma$ are left out.

**2. A super-ob is a mean, so its footprint is a mean.** The super-ob value is $y = \frac{1}{N}\sum_k y_k$. The
prediction is linear in the tile values, so the prediction consistent with $y$ is

$$\frac{1}{N}\sum_k H_k x = \sum_t \Big(\frac{1}{N}\sum_k \hat w_{k,t}\Big)\, x_t .$$

The exact footprint is the **average of the raw footprints**: a mixture of $N$ Gaussians,
$m(\mathbf{r}) = \frac{1}{N}\sum_k \mathcal{N}(\mathbf{r};\, \mathbf{x}_k,\, \sigma^2 I)$ (panel b).

**3. The mean and covariance of that mixture.**

- Mean: $\boldsymbol{\mu} = \frac{1}{N}\sum_k \mathbf{x}_k$, the mean position of the raw obs, which is where the reader
  already places the super-ob.
- Covariance, by the law of total covariance (average spread *within* each Gaussian + spread *between* their centres):

$$C = \underbrace{\sigma^2 I}_{\text{one raw footprint}} \;+\; \underbrace{S}_{\text{spread of the raw obs}}, \qquad S = \frac{1}{N}\sum_k (\mathbf{x}_k-\boldsymbol{\mu})(\mathbf{x}_k-\boldsymbol{\mu})^{\mathsf T}.$$

**4. The new operator (moment matching).** Replace the mixture by the single Gaussian with the same mean and covariance,
$\mathcal{N}(\mathbf{r};\, \boldsymbol{\mu},\, C)$:

$$\hat w_t \propto A_t \exp\!\left(-\tfrac12 q_t\right), \qquad q_t = (\mathbf{c}_t-\boldsymbol{\mu})^{\mathsf T} C^{-1} (\mathbf{c}_t-\boldsymbol{\mu}), \qquad \text{keep } q_t \le 4 .$$

$q_t \le 4$ is "within 2 standard deviations", the same cut as raw mode's $2\sigma$; the search radius is
$2\sqrt{\lambda_{\max}(C)}$.

**5. Special cases.**

| raw obs in the cell | S | new footprint |
|---|---|---|
| 1 | 0 | exactly the raw-obs footprint (σ = 12.5 km) |
| a few, close together | small | slightly wider, stretched along the obs layout |
| a full, uniformly covered cell of size s | ≈ (s·k)²/12 per axis | σ ≈ 14.8 km at s = 0.25°, ≈ 20 km at 0.5° |
| obs far apart (two clusters) | large | the mixture becomes bimodal; one Gaussian smooths over the gap (figure below, c) |

**6. In the code** (`9624356`). The reader accumulates, per cell, the offsets of the raw obs from the cell centre and
their squares and cross-products, then $S$ = second moments − mean², converted to km² at the super-ob's mean latitude
$\bar\varphi$: $S_{xx} = \mathrm{var}_{\text{lon}}\,(k\cos\bar\varphi)^2$, $S_{yy} = \mathrm{var}_{\text{lat}}\,k^2$,
$S_{xy} = \mathrm{cov}\,k^2\cos\bar\varphi$, with $k$ = km per degree. `get_obs_pred` builds $C$, searches out to
$2\sqrt{\lambda_{\max}}$ and weights by $\exp(-\frac12 q_t)\,A_t$.

**The old operator** used a uniform disk of radius $s/\sqrt\pi$ degrees (the cell's area) for every super-ob. It ignores
both the raw footprint $\sigma$ and the layout $S$.

Note: the exact target used in this notebook averages the *normalised* raw-obs tile weights (step 2). Where all raw
footprints see the same land, that equals sampling the mixture at the tile centres; near coasts it can differ slightly.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(19, 5.8), constrained_layout=True)
sg = FOV_RAW_KM

# (a) 1D: averaging Gaussians adds the spread of their centres to the variance
x1 = np.linspace(-60, 60, 1201)
xk = np.array([-9.0, 1.0, 11.0])
mu, s2 = xk.mean(), xk.var()
n1 = lambda x, m, v: np.exp(-0.5 * (x - m) ** 2 / v) / np.sqrt(2 * np.pi * v)
ax = axs[0]
for k, m in enumerate(xk):
    ax.plot(x1, n1(x1, m, sg ** 2) / len(xk), color=cols[k], lw=1.5, ls='--')
    ax.plot(m, 0, 'o', color=cols[k], mec='k', ms=8, clip_on=False, zorder=5)
mix1 = np.mean([n1(x1, m, sg ** 2) for m in xk], axis=0)
mm1 = n1(x1, mu, sg ** 2 + s2)
ax.plot(x1, mix1, 'k', lw=2.5, label='mixture: mean of the Gaussians')
ax.plot(x1, mm1, color='darkgreen', lw=2, ls='-.', label=f'N(μ, σ² + s²), √(σ²+s²) = {np.sqrt(sg**2 + s2):.1f} km')
h = mm1.max()
for y, w, lab, c in [(0.62 * h, sg, f'σ = {sg} km (one raw footprint)', cols[1]),
                     (0.40 * h, np.sqrt(s2), f's = {np.sqrt(s2):.1f} km (spread of the obs)', 'tab:red'),
                     (0.18 * h, np.sqrt(sg ** 2 + s2), f'√(σ²+s²) = {np.sqrt(sg**2 + s2):.1f} km', 'darkgreen')]:
    ax.annotate('', xy=(mu + w, y), xytext=(mu, y), arrowprops=dict(arrowstyle='<->', color=c, lw=1.8))
    ax.text(31, y, lab, color=c, va='center', fontsize=9,
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.85, pad=1))
ax.axvline(mu, color='0.5', lw=0.8)
ax.set_ylim(0, 1.15 * mix1.max()); ax.set_yticks([])
ax.set_xlabel('distance [km]'); ax.legend(loc='upper left', fontsize=9)
ax.set_title('(a) 1D: averaging N Gaussians (dashed, x 1/N)\nvariance of the mixture = σ² + s²')

# (b) 2D: C = sigma^2 I + S, as covariance ellipses (1 sigma), for the 3-obs super-ob above
ax = axs[1]
for C_, c, lab in [(sg ** 2 * np.eye(2), cols[1], 'σ²I: one raw footprint'),
                   (S2 + 1e-9 * np.eye(2), 'tab:red', 'S: spread of the raw obs'),
                   (C2, 'darkgreen', 'C = σ²I + S: new footprint')]:
    e = ellipse_xy(C_, 1)
    ax.plot(e[:, 0], e[:, 1], color=c, lw=2.2, label=lab)
ax.plot(P[:, 0], P[:, 1], 'o', color='w', mec='k', ms=8, zorder=5)
ax.plot(0, 0, '*', color='w', mec='k', ms=15, zorder=6)
ax.set_aspect('equal'); ax.set_xlim(-22, 22); ax.set_ylim(-22, 22)
ax.set_xlabel('east [km]'); ax.set_ylabel('north [km]'); ax.legend(loc='lower right', fontsize=9)
ax.set_title('(b) 2D: covariances add (1σ ellipses)\nsuper-ob of 3 raw obs from the figures above')

# (c) how good is one Gaussian? two raw obs a distance d apart (the hardest case: most bimodal)
dd = np.linspace(0, 60, 61)
ovl = []
for d in dd:
    Pd = np.array([[-d / 2, 0.0], [d / 2, 0.0]])
    mixd = np.mean([gauss(sg ** 2 * np.eye(2), x, y) for x, y in Pd], axis=0)
    newd = gauss(sg ** 2 * np.eye(2) + np.cov(Pd.T, bias=True))
    ovl.append(np.minimum(mixd, newd).sum() / mixd.sum())
ax = axs[2]
ax.plot(dd, ovl, 'k', lw=2.2, label='2 raw obs, d apart')
d_cell = np.hypot(SUPEROB_GRID * KM_PER_DEG * np.cos(np.radians(LAT_DEMO)), SUPEROB_GRID * KM_PER_DEG)
for x, lab, y0 in [(12.5, 'H121 grid spacing (12.5 km)', 0.82), (d_cell, f'{SUPEROB_GRID}° cell diagonal ({d_cell:.0f} km)', 0.885)]:
    ax.axvline(x, color='0.5', ls='--', lw=1)
    ax.text(x + 0.8, y0, lab, rotation=90, va='bottom', fontsize=9, color='0.3')
ax.set_xlabel('distance between the two raw obs, d [km]'); ax.set_ylabel('overlap ∫ min(mixture, Gaussian)')
ax.set_ylim(0.8, 1.005); ax.grid(alpha=0.3); ax.legend(loc='lower left')
ax.set_title('(c) one Gaussian vs the exact mixture\n(1 = identical)')
fig.savefig(FIG_DIR / 'summary_math.png', dpi=200, bbox_inches='tight')
plt.show()

## Acceptance checks

In [ ]:
CHECKS = []
def check(name, status, detail=''):
    CHECKS.append({'check': name, 'result': status, 'detail': detail})

check('1. S (reader formula) == direct covariance, < 1e-6 km^2', 'PASS' if S_DIFF < 1e-6 else 'FAIL', f'{S_DIFF:.1e}')

single = SO[SO.emitted_old & (SO.n_raw == 1)]
Wn1 = fov_N(single.lat, single.lon, single[['Sxx', 'Syy', 'Sxy']].to_numpy())
Wr1 = fov_R(single.lat, single.lon)
d1 = abs(Wn1 - Wr1).max() if len(single) else 0.0
check('2. single-ob super-obs: N == R (max |Δŵ| < 1e-12)', 'PASS' if d1 < 1e-12 else 'FAIL',
      f'{len(single):,} super-obs, max {d1:.1e}')

# 3. kept tiles have q <= 4, and no tile with q <= 4 is missing (brute force over a 1.5x larger search)
Sr = SOR[['Sxx', 'Syy', 'Sxy']].to_numpy()
_, obs_k, tile_k, q_k = fov_N(SOR.lat, SOR.lon, Sr, return_q=True)
cxx, cyy, cxy, det, lmax = footprint_C(Sr)
la_, lo_ = SOR.lat.to_numpy(np.float64), SOR.lon.to_numpy(np.float64)
cand = TREE.query_ball_point(np.c_[lo_, la_], r=1.5 * FAC_SEARCH * np.sqrt(lmax) * KM2DEG / np.cos(np.radians(la_)))
missing = 0
kept_sets = pd.Series(tile_k[q_k <= 4]).groupby(obs_k[q_k <= 4]).apply(set)
for n, c in enumerate(cand):
    c = np.array(c)
    dx = (COM_LON[c] - lo_[n]) * KM_PER_DEG * np.cos(np.radians(la_[n])); dy = (COM_LAT[c] - la_[n]) * KM_PER_DEG
    q = (cyy[n] * dx ** 2 - 2 * cxy[n] * dx * dy + cxx[n] * dy ** 2) / det[n]
    missing += len(set(c[q <= 4]) - kept_sets.get(n, set()))
nnz_ok = all(W_N[n].indices.size == len(kept_sets.get(n, set())) for n in range(len(SOR)))
check('3. N footprints: kept tiles q <= 4, none with q <= 4 missing', 'PASS' if missing == 0 and nnz_ok else 'FAIL',
      f'{len(SOR):,} super-obs, missing {missing}')

sums = np.concatenate([np.asarray(W.sum(1)).ravel() for W in (W_X, W_N, W_O, W_RM, *WG.values())])
sums = sums[sums > 0]
check('4. ŵ sums to 1 for every footprint', 'PASS' if np.allclose(sums, 1, atol=1e-12) else 'FAIL',
      f'max |sum-1| {np.abs(sums - 1).max():.1e}')
check('5. (O), (R) reproduce ObsFcstAna fcst', 'not run', 'needs Discover ObsFcstAna + SFMC_FCST (Discover notebook §6c)')
check('6. per-file QC counts match ldas_log', 'not run', 'needs the Discover case files and ldas_log')
elapsed = time.time() - T_START
check('7. runtime < ~3 min', 'PASS' if elapsed < 180 else 'FAIL', f'{elapsed:.0f} s')
print(pd.DataFrame(CHECKS).to_string(index=False))